# VATEX - adnotacje gotowe

Buduje szkielet znaczników i plik zapytań części testowej VATEX z pobranych klipów.

Żadna decyzja konfiguracyjna nie zapada na tym pliku: zapadają one wyłącznie na odcinkach deweloperskich obu seriali, a część testowa VATEX jest oceniana dopiero na zamrożonych konfiguracjach, jako kontrola przenoszenia wniosków na inną domenę. Część deweloperska VATEX (`vatex_01_dev.ipynb`) służy tylko wyznaczeniu progu dopasowania fraz do nazw klas i kontroli sygnału ruchu, więc do tego notatnika nie wchodzi. Plik kontrolny `vatex_check.jsonl` powstaje w `notebooks/vatex_check.ipynb`, czyli tam, gdzie leży test, któremu służy.

**Wymaga:** `vatex_02_test_acquisition.ipynb` oraz skryptu `scripts/prepare_data/vatex_leak_filter.py`.

**Zapisuje:**

| plik | co zawiera |
|---|---|
| `data/annotations/vatex/vatex_query_tags.csv` | znaczniki wymagań i etykieta złożoności; dopisywane są tylko brakujące wiersze |
| `data/annotations/vatex/vatex_queries_test.jsonl` | zapytania zbioru testowego, jeden opis na klip |

**Dalej:** eksperymenty, `notebooks/README.md`.

In [ ]:
SOURCE = "vatex"
CONFIG = "configs/vatex_base.yaml"

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.annotation import tags
from src.utils import config as conf
from src.utils import vatex
from src.utils.queries import load_jsonl, save_jsonl

cfg = conf.load(conf.path(CONFIG))

QUERIES_JSONL   = conf.path(cfg["paths"]["queries_test"])
ANNOTATIONS_DIR = QUERIES_JSONL.parent
TAGS_CSV = tags.path_for(SOURCE, ANNOTATIONS_DIR)
TAGS_NEW = TAGS_CSV.with_name(TAGS_CSV.stem + "_new.csv")
# VATEX assigns a narrower set than the series -- three tags and the complexity
# label; the block below says why
COLUMNS = tags.columns_for(SOURCE)


def check_alignment(rows, records):
    """Stops the cell when a desc_id in the tag file now names a different clip.

    The file is keyed by desc_id, so a drift would carry hand-filled tags onto
    other queries without a word. It must not be resolved either way here.
    """
    drift = tags.misaligned(rows, {q.desc_id: q.vid_name for q in records})
    if drift:
        listing = "\n".join(f"   {desc_id}: file says {was}, now {now}"
                            for desc_id, was, now in drift[:10])
        raise ValueError(
            f"{TAGS_CSV.name} has rows naming a different clip than their"
            f" desc_id does now ({len(drift)}) - the tags would land on other"
            f" queries. Check what changed in the clip set:\n{listing}")


report       = vatex.load_report(conf.path(cfg["paths"]["report"]))
descriptions = vatex.load_descriptions(conf.path(cfg["paths"]["descriptions"]))
split        = vatex.load_split(conf.path(cfg["paths"]["split"]))

clips_dir = conf.path(cfg["paths"]["clips"])
ok_vids   = vatex.ok_clips(report, clips_dir)                    # implementation check
test_vids = vatex.test_split_clips(report, split, clips_dir)     # experiments

tag_rows = tags.load(TAGS_CSV, COLUMNS)

print(f"ok clips:   {len(ok_vids):>5}")
print(f"test split: {len(test_vids):>5}  (K400 leak filtered out: {len(ok_vids) - len(test_vids)})")
print(f"tag file {len(tag_rows)} rows"
      f"{'' if TAGS_CSV.exists() else ' (no ' + TAGS_CSV.name + ' yet)'}")

## 1. Znaczniki zapytań

**Zapisuje:** `data/annotations/vatex/vatex_query_tags_new.csv` - jeden wiersz na zapytanie testowe, z kolumnami na trzy znaczniki wymagań i etykietę złożoności. Plik nie jest nadpisywany: blok dopisuje brakujące wiersze i odświeża kolumny kontekstowe (klip, split, treść zapytania), ale nigdy nie rusza komórki, która została wypełniona.

**Zestaw znaczników jest węższy niż w serialach i to jest decyzja rozdziału 4.** Zapytaniu VATEX przypisuje się tylko to, co da się rozstrzygnąć z samego jednozdaniowego opisu klipu: `wymaga_obiektu`, `wymaga_scenerii`, `wymaga_ruchu` i złożoność. `wymaga_osoby` trzeba by czytać na tle pozostałych zdarzeń tego samego nagrania, a klipy VATEX są od siebie niezależne; `wymaga_mimiki` wymaga informacji o wyrazie twarzy, której opisy nie zawierają. Brak kolumny znaczy "tego się tutaj nie przypisuje", a nie `0`. Odpada też `identities` - VATEX nie ma powracających postaci z profilami.

**Identyfikator zapytania pochodzi z nazwy klipu**, nie z pozycji na liście (`src/utils/vatex.py::desc_id_for`). Numeracja po pozycji przesunęłaby wszystkie identyfikatory, gdyby ze zbioru wypadł jeden klip albo zmienił się filtr przecieku, a plik znaczników jest kluczowany po `desc_id`: ręcznie wpisane znaczniki przykleiłyby się wtedy po cichu do innych zapytań. Niezależnie od tego blok porównuje kolumnę `episode` wczytanego pliku z klipem, na który dany `desc_id` wskazuje teraz, i przy rozjeździe przerywa.

In [ ]:
records  = vatex.build_queries(test_vids, descriptions)
existing = tags.load(TAGS_CSV, COLUMNS)
check_alignment(existing, records)

tag_rows, tag_counts = tags.skeleton(vatex.tag_source_rows(records), existing,
                                     columns=COLUMNS)
tags.save(tag_rows, TAGS_NEW, COLUMNS)

problems = tags.validate(tag_rows, COLUMNS)
for problem in problems[:10]:
    print(f"  INVALID {problem}")
if problems:
    print(f"{len(problems)} invalid cells - fix them before the queries are rebuilt")

print("columns: " + "; ".join(COLUMNS))
print(f"rows: {len(tag_rows)}  (added {tag_counts['added']},"
      f" kept {tag_counts['kept']}, dropped {tag_counts['dropped']})")
if tag_counts["retexted"]:
    print(f"WARNING: wording changed for {len(tag_counts['retexted'])} rows -"
          " their tags were assigned to a different query")

coverage = tags.coverage(tag_rows, columns=COLUMNS)
print(f"filled: {coverage['filled']} of {coverage['total']}")
print(f"saved -> {TAGS_NEW.relative_to(ROOT)}")
print(f"copy it over {TAGS_CSV.name} by hand once you are done")

## 2. Plik zapytań

**Zapisuje:** `data/annotations/vatex/vatex_queries_test.jsonl` - zbiór testowy po odfiltrowaniu przecieku treningowego Kinetics-400, po jednym opisie na klip, w schemacie wspólnym dla wszystkich trzech zbiorów danych. To jest wejście eksperymentów.

`vid_name` wskazuje klip, `ts` to jego granice na własnej osi czasu (klip jest już wycinkiem, więc zaczyna się od zera), a `event_id` to identyfikator klipu, bo poprawną odpowiedzią jest ten właśnie klip. Lista `identities` zostaje pusta.

Blok czyta plik znaczników z dysku, nie z poprzedniego bloku, więc po ręcznym uzupełnieniu `vatex_query_tags.csv` można uruchomić sam ten. Gdy pliku jeszcze nie ma, zapytania powstają bez znaczników i blok to wypisuje.

In [ ]:
# The tag file is read HERE, not taken from the cell above: after filling it in
# by hand you run this cell alone, and it has to see what is in the file now.
records  = vatex.build_queries(test_vids, descriptions)
tag_rows = tags.load(TAGS_CSV, COLUMNS)
check_alignment(tag_rows, records)

if not tag_rows:
    print(f"no {TAGS_CSV.name} yet - queries are written without tags;"
          " copy the _new file over it and run this cell again\n")
counts = tags.merge(records, tag_rows, columns=COLUMNS)

written = save_jsonl(records, QUERIES_JSONL)

identifiers = [q.desc_id for q in records]
print(f"queries: {len(records)} (expected {len(test_vids)})")
print(f"desc_id unique: {len(identifiers) == len(set(identifiers))}")
print(f"with tags: {counts['tagged']}, without: {counts['untagged']},"
      f" wording corrected: {counts['retexted']}")
print(f"saved -> {written.relative_to(ROOT)}")

example = load_jsonl(written)
print(f"\nexample record:\n{example[0].to_dict() if example else '(no queries yet)'}")

## 3. Statystyka zapytań

Bez podziału na dev i test - zbiór ma wyłącznie część testową. Liczebności rozstrzygają, które przekroje da się analizować: znacznik występujący w kilkunastu zapytaniach nie utrzyma przedziału ufności i wchodzi dalej jako wielkość efektu.

Wierszy z postaciami tu nie ma, bo VATEX nie przypisuje `identities`, i nie ma wierszy `wymaga_osoby` ani `wymaga_mimiki`.

In [ ]:
from src.evaluation import tables

records  = vatex.build_queries(test_vids, descriptions)
tag_rows = tags.load(TAGS_CSV, COLUMNS)
tags.merge(records, tag_rows, columns=COLUMNS)

stats = tags.statistics(records, COLUMNS)


def share(value):
    """Count with its share of the set, or "-" when the set is empty."""
    if not stats["count"]:
        return "-"
    return f"{value:>5}  ({100 * value / stats['count']:>4.1f}%)"


table = [["liczba zapytan", str(stats["count"])]]
table += [[tag, share(count)] for tag, count in stats["requirements"].items()]
table += [[name, share(stats["complexity"][value])]
          for value, name in (("P", "proste (P)"), ("Z", "zlozone (Z)"))]

tables.show(f"Zapytania {SOURCE} wedlug znacznikow",
            ["Wielkosc", "test"], table,
            note="Udzial liczony wzgledem liczby zapytan. VATEX ma wylacznie "
                 "czesc testowa. Znacznikow wymaga_osoby i wymaga_mimiki sie tu "
                 "nie przypisuje - jednozdaniowy opis klipu ich nie rozstrzyga.")

if stats["complexity_missing"]:
    print()
    print(f"zapytan bez etykiety zlozonosci: {stats['complexity_missing']}"
          " - uzupelnij plik znacznikow")